# 📘 Lesson 19: Full Decoder Transformer (GPT Architecture in PyTorch)

**Step-by-Step Interactive Homework Notebook** with modular code execution and detailed explanations.


### 🔹 Step 1

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import os
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

torch.manual_seed(42)
random.seed(42)


### 🔹 Load the Dataset

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
DATA_PATH = "/Users/mac/Desktop/Machine Learning/DL/DB/Names/names.txt"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    names=[ line.strip().lower() for line in f if line.strip() ]
    

print("Number of names:", len(names))
print("First 10 names:")

for name in names[:10]:
    print(name)
    
names = [f".{name}." for name in names]


### 🔹 Build the Vocabulary

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
vocab=sorted(set("".join(names)))

stoi={ch:i for i, ch in enumerate(vocab)}
itos={i:ch for ch, i in stoi.items()}

VOCAB_SIZE = len(vocab)

print("Vocabulary:", vocab)
print("Vocabulary size:", VOCAB_SIZE)


### 🔹 Encode
Decode

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def encode(text):
    return [stoi[ch] for ch in text]

def decode(idx):
    return "".join(itos[i] for i in idx)

example = names[0]

encoded = encode(example)

print("Original:", example)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))


### 🔹 Train Test Split
Custom Dataset

**Purpose**: Dataset Preprocessing & Feature Scaling.

- Splits data into Training/Validation/Testing sets to evaluate generalization.

- Standardizes features ($\mu=0, \sigma=1$) to stabilize gradient descent and prevent vanishing/exploding updates.


In [ ]:
train_names, test_names=train_test_split(
    names, test_size=0.2,random_state=42
)

print("Training names:", len(train_names))
print("Testing names:", len(test_names))

class NamesDataset(Dataset):
    def __init__(self, names):
        self.names=names
        
    def __len__(self):
        return len(self.names)
    
    def __getitem__(self, idx):
        name=self.names[idx]
        
        encoded=torch.tensor(
            encode(name),
            dtype=torch.long
        )
        
        x=encoded[:-1]
        y=encoded[1:]
        
        return x, y
    
train_dataset = NamesDataset(train_names)
test_dataset = NamesDataset(test_names)


### 🔹 Check one

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
x, y = train_dataset[0]

print("X:", x)
print("Y:", y)

print("X decoded:", decode(x.tolist()))
print("Y decoded:", decode(y.tolist()))


### 🔹 Dataloader

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

x, y = next(iter(train_loader))

print("X shape:", x.shape)
print("Y shape:", y.shape)


### 🔹 Hyperparameters
Create the RNN Class

**Purpose**: Model Architecture Definition.

- Defines the network structure, layer projections, activations, and the forward propagation computation graph.


In [ ]:
EMBEDDING_DIM = 16
HIDDEN_SIZE = 64

class CharRNN(nn.Module):
    def __init__(
        self, 
        vocab_size,
        embedding_dim,
        hidden_size
    ):
        super().__init__()
        
        self.E=nn.Parameter(
            torch.randn(
                vocab_size, embedding_dim
            )*0.01
        )
        
        self.hidden_size=hidden_size


### 🔹 Input-> Hidden
Hidden -> hidden

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
self.W_xh=nn.Parameter(
            torch.randn(
                embedding_dim, hidden_size
            )*0.01
        )
        
        self.W_hh=nn.Parameter(
            torch.randn(
                hidden_size, hidden_size
            )*0.01
        )


### 🔹 Hidden bias
Hidden->Output

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
self.b_h=nn.Parameter(
            torch.zeros(hidden_size)
        )
        
        self.W_hy=nn.Parameter(
            torch.randn(
                hidden_size,vocab_size
            )*0.01
        )


### 🔹 Output bias
Convert character IDs into embeddings
Initial hidden state

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
self.b_y=nn.Parameter(
            torch.zeros(vocab_size)
        )
    def forward(self, x):
        batch_size, sequence_length=x.shape
        
        embeddings = self.E[x]
        
        h = torch.zeros(
            batch_size,
            self.hidden_size,
            device=x.device
        )
        
        outputs=[]
        for t in range(sequence_length):


### 🔹 Current character embedding
Recurrent equation
Hidden->output

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
x_t=embeddings[:,t,:]
            
            h=torch.tanh(
                x_t @ self.W_xh+h @ self.W_hh + self.b_h
            )
            
            logits=(
                h @ self.W_hy+self.b_y
            )
            
            outputs.append(logits)


### 🔹 Combine all time-step outputs
Create the Model

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
outputs=torch.stack(
            outputs, dim=1
        )
        return outputs
    


model=CharRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE
).to(device)

print(model)


### 🔹 Model Parameters
Loss function & Optimizer

**Purpose**: Loss Function & Optimizer Initialization.

- Configures optimization objective and update rule (e.g., Adam, SGD with momentum, weight decay).


In [ ]:
for name, param in model.named_parameters():
    print(name, param.shape)
        
        
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


### 🔹 Test one forward pass

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
x, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

logits = model(x)

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Output shape:", logits.shape)


### 🔹 Training loop
Training

**Purpose**: Training & Optimization Loop.

- **Forward Pass**: Compute model predictions and loss.

- **Backward Pass**: `loss.backward()` calculates gradients via automatic differentiation.

- **Optimizer Step**: `optimizer.step()` updates trainable weights; `optimizer.zero_grad()` clears gradients.


In [ ]:
EPOCHS=20
train_losses=[]
test_losses=[]

for epoch in range(EPOCHS):
    
    model.train()
    total_train_loss=0.0
    
    for x, y in train_loader:
        x=x.to(device)
        y=y.to(device)
        
        optimizer.zero_grad()
        logits=model(x)
        
        loss=criterion(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        )
        loss.backward()
        optimizer.step()
        
        total_train_loss+=loss.item()
        
    avg_train_loss=(
        total_train_loss/len(train_loader)
    )


### 🔹 Evaluation

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model.eval()
    
    total_test_loss=0.0
    
    with torch.no_grad():
        for x, y in test_loader:
            x=x.to(device)
            y=y.to(device)
            
            logits=model(x)
            
            loss=criterion(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )
            total_test_loss+=loss.item()
            
    avg_test_loss=(
        total_test_loss/len(test_loader)
    )


### 🔹 Save losses

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    
    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"| Train Loss: {avg_train_loss:.4f} "
        f"| Test Loss: {avg_test_loss:.4f}"
    )
    
    
model.eval()
x, y=test_dataset[0]
x=x.unsqueeze(0).to(device)

with torch.no_grad():
    logits=model(x)
    
predictions=logits.argmax(dim=-1)

print("Input:      ", decode(x[0].cpu().tolist()))
print("Target:     ", decode(y.tolist()))
print("Prediction: ", decode(predictions[0].cpu().tolist()))


### 🔹 Plot the Losses

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,label="Train Loss"
)

plt.plot(
    test_losses,
    label="Test Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Character-level RNN Training")

plt.legend()
plt.show()


### 🔹 Generation Function
Start with "."
Initial hidden state

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def generate_name(model, max_length=20):
    model.eval()
    
    current_char=stoi["."]
    
    h=torch.zeros(
        1, model.hidden_size,
        device=device
    )
    generated=[]
    
    with torch.no_grad():
        for _ in range(max_length):


### 🔹 Current character ->tensor
Character ->embedding
Recurrent step

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
x=torch.tensor(
                [[current_char]],
                dtype=torch.long,
                device=device
            )
            
            x_t=model.E(x[:,0])
            
            h=torch.tanh(
                x_t @ model.W_xh+h@ model.W_hh+model.b_h
            )
            #Hidden -> output
            logits=(
                h @ model.W_hy+model.b_y
            )


### 🔹 Logits->probabilities
Sample next character
Stop at "."

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
probs=torch.softmax(
                logits, dim=-1
            )
            next_char=torch.multinomial(
                probs, num_samples=1
            ).item()
            
            if next_char==stoi["."]:
                break
            
            generated.append(
                itos[next_char] # type:ignore
            )


### 🔹 Feed prediction back into the RNN

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
current_char=next_char
            
    return "".join(generated)


for _ in range(20):
    print(generate_name(model))


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
